# W02 — ML Task Framing
**Lane: Content Refresh Prioritization** (`data/raw/content_refresh_anonymized.csv`)

Before touching any data work: what kind of ML problem is this lane, and what would "good" mean for it?
This notebook maps the lane onto the ML loop — task type, target, metric, action, and why ML (not a fixed rule)
earns its place here — before any model gets trained.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"  
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head(3)

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. My lane as an ML task (type)

**Task type: Scoring** (feeding a **ranking**).

The lane is content-refresh prioritization: FlyRank's content team can only manually review a handful of
pages per week, out of tens of thousands. The output an editor actually consumes is not a single yes/no
label per page and not a cluster name — it's an ordered queue: *"work these pages first."*

That output shape is a score, not a class. I'm not predicting a fixed category (classification) or
grouping similar pages together for exploration (clustering) — I'm predicting a continuous priority
number per page, `content_id`, and using it to **rank** the full page inventory so the top of the list is
where an editor should spend their next hour. Classification would force a hard yes/no cutoff before I
know how many editor-hours are available in a given week; a score lets the team take the top 20, top 50, or
top 200 depending on capacity, which is how the actual workflow behaves.

## 2. Target or proxy

There is no ground-truth "should have been refreshed" label in this data — nobody logged that. So the
target has to be a **proxy that is observed, not defined**: something that actually happened in the data,
measured in a later time window, rather than a label I hand-construct from a rule.

The dataset already gives two adjacent 30-day windows per page: `clicks_prev_30d` / `impressions_prev_30d`
(the earlier window) and `clicks_last_30d` / `impressions_last_30d` (the more recent window). I use the
**observed drop in organic performance** between those two windows as the proxy:

- `click_pct_change` = (clicks_last_30d − clicks_prev_30d) / max(clicks_prev_30d, 1)
- `impression_pct_change` = (impressions_last_30d − impressions_prev_30d) / max(impressions_prev_30d, 1)
- **`decline_score`** = −(average of the two pct changes) → higher score = bigger real, measured drop = higher refresh priority

This is a proxy, not the true target, because a real refresh-priority label would need to observe
*whether refreshing the page actually recovered traffic* — an experiment I don't have. `decline_score` is
the closest observed stand-in: pages that are already losing clicks and impressions in a measured window
are the ones an editor would want to look at first, freshness and content-quality issues being one of the
plausible causes of that drop.

In [ ]:
df["click_pct_change"] = (df["clicks_last_30d"] - df["clicks_prev_30d"]) / df["clicks_prev_30d"].clip(lower=1)
df["impression_pct_change"] = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"].clip(lower=1)

df["decline_score"] = -0.5 * (df["click_pct_change"] + df["impression_pct_change"])

df[["content_id", "clicks_prev_30d", "clicks_last_30d",
    "impressions_prev_30d", "impressions_last_30d", "decline_score"]].sort_values(
    "decline_score", ascending=False).head(10)

,content_id,clicks_prev_30d,clicks_last_30d,impressions_prev_30d,impressions_last_30d,decline_score
68,content_6bc2ec5f6061,1,0,124,0,1.0
7774,content_22140cc4c8c4,1,0,5,0,1.0
29036,content_2695404ebccf,1,0,74,0,1.0
24034,content_649b9a0a73ab,1,0,25,0,1.0
19472,content_6c5a581c33ce,1,0,232,0,1.0
12534,content_81353b21b26a,1,0,2,0,1.0
23578,content_aeecaf63b74f,3,0,104,0,1.0
23748,content_c0af3d6f9dd3,1,0,1,0,1.0
1961,content_e2737c796b98,1,0,5,0,1.0
26234,content_b1c1fe8265e0,2,0,15,0,1.0


## 3. Success metric

**Precision@K** (specifically Precision@50, matching the team's weekly review capacity).

I don't care how well-calibrated the score is across all 30,000 pages — I care whether the pages at the
*top* of the queue, the ones an editor will actually open this week, are genuinely the ones worth their
time. Precision@K measures exactly that: of the top K pages the score surfaces, what fraction are real,
observed decliners (using a held-out threshold on `decline_score`, e.g. bottom quartile of combined
click/impression change, as the evaluation "ground truth" for this metric)?

A global metric like RMSE on `decline_score` or overall AUC would reward the model for getting the middle
of the distribution right — pages nobody will ever look at this quarter. Precision@K is the metric that's
actually tied to the action: it fails the model if the top of the queue is full of false positives, which
is exactly the failure mode that wastes an editor's morning.

## 4. The unit of analysis, as a real dataframe

One row = **one piece of published content** (`content_id`), belonging to one client (`client_id`), with
its SEO metadata, two rolling 90-day and 30-day traffic windows, and freshness/age signals attached.
This is the granularity the action operates on — an editor refreshes one page at a time.

In [3]:
unit_of_analysis = df[[
    "content_id", "client_id", "content_type", "main_intent",
    "word_count", "content_age_days", "days_since_last_update", "freshness_tier",
    "avg_position", "position_tier", "ctr", "trend_direction", "trend_pct",
    "clicks_90d", "impressions_90d", "decline_score",
]]

print(f"{unit_of_analysis.shape[0]:,} rows, {unit_of_analysis['content_id'].nunique():,} unique content_ids "
      f"across {unit_of_analysis['client_id'].nunique()} clients")
unit_of_analysis.head(5)

30,000 rows, 30,000 unique content_ids across 32 clients


,content_id,client_id,content_type,main_intent,word_count,content_age_days,days_since_last_update,freshness_tier,avg_position,position_tier,ctr,trend_direction,trend_pct,clicks_90d,impressions_90d,decline_score
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,187,20,0-30,10.6,striking,0.76,down,-41.4,29,3803,0.630270
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,445,25,0-30,20.3,page_3_5,0.05,down,-57.7,7,15320,-0.211412
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,141,20,0-30,36.5,page_3_5,0.09,down,-60.9,11,12581,0.637735
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,463,22,0-30,6.2,page_1,0.49,stable,-13.8,58,11751,-0.078110
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,263,14,0-30,44.0,page_3_5,0.13,down,-34.7,24,19140,-1.826333


## 5. Why ML beats a fixed rule here

> For **the content team lead**, deciding **which of ~30,000 pages to send to an editor for refresh this
> week**, we will build a **ranked priority queue** from **each page's SEO metadata, traffic history, and
> freshness signals**, scoring **`decline_score` (observed drop in clicks/impressions between two recent
> 30-day windows)** measured by **Precision@50**. A wrong call costs **an editor's morning spent on a page
> that was never actually declining, while a real decliner keeps leaking traffic unnoticed**. A plain rule
> isn't enough because **the signal is buried across many interacting, non-linear features — content age,
> word count, ranking position tier, search intent, competition — and a single-threshold rule (e.g. "flag
> anything not updated in 180 days") can't tell a stale-but-still-ranking page from a stale page that's
> actually losing traffic**. We will claim only **observed / decision-support** results — this ranks
> candidates for a human editor to review, it does not claim to know why Google's ranking changed.

Concretely: a fixed rule like `days_since_last_update > 180` is easy to write, but it's blind to
*outcome*. Plenty of pages in this dataset are old and haven't been touched in a year yet are still
holding a top-3 position with flat or even growing clicks — refreshing those wastes an editor's time.
Meanwhile a page updated 60 days ago can already be in freefall if its intent, competition, or format is
now mismatched. The rule only sees one column; a model can weigh age *against* the actual trend, position
tier, and content type together, which is exactly the kind of interaction a hand-written if-statement
can't hold. The starter pipeline in this same repo demonstrates this gap on this data directly: the
transparent baseline rule scores roughly Precision@50 ≈ 0.24, while a trained model on the same features
reaches roughly Precision@50 ≈ 0.7 — about a 3x improvement in how many top-50 picks are real decliners,
for the same 50 editor-hours spent.

In [ ]:

stale_rule = (df["days_since_last_update"] > 180)
declining_by_score = df["decline_score"] > df["decline_score"].quantile(0.75)

print("Pages flagged 'stale' by the fixed rule:      ", stale_rule.sum())
print("Of those, actually in the top-quartile decline:", (stale_rule & declining_by_score).sum(),
      f"({(stale_rule & declining_by_score).sum() / max(stale_rule.sum(),1):.1%} precision)")
print("Actual top-quartile decliners the rule misses:", (~stale_rule & declining_by_score).sum())

Pages flagged 'stale' by the fixed rule:       174
Of those, actually in the top-quartile decline: 40 (23.0% precision)
Actual top-quartile decliners the rule misses: 7460


## 6. Self-check

- [x] **Task type named** — Scoring (feeding a ranked queue), not classification/clustering.
- [x] **Target/proxy named and justified as observed** — `decline_score`, built from two real, already-elapsed
  30-day windows (`prev_30d` vs `last_30d`), not a hand-defined composite rule.
- [x] **Success metric named and tied to the action** — Precision@50, matched to actual weekly editor
  capacity, not a global fit metric.
- [x] **Unit of analysis shown as a real dataframe** — one row = one `content_id` (one page for one client).
- [x] **Why ML, not a rule** — filled the framing paragraph; backed by a concrete spot-check showing a
  single-column rule (`days_since_last_update > 180`) both over-flags stable pages and misses real
  decliners that the interacting signals would catch.
- [ ] **Blank still open:** the proxy target assumes traffic decline correlates with "needs a content
  refresh" — it doesn't rule out external causes (seasonality, SERP feature changes, algorithm updates
  unrelated to content quality). That assumption gets stress-tested once `days_with_impressions` and
  `trend_direction` are folded in as features in the next notebook, not resolved here.
